# 07 - State-of-the-Art Sentiment Analysis with DeBERTa-v3
## Pushing IMDb Classification Accuracy to 95%+

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ranjitcodes1/IMDB-Sentiment-Research-Project/blob/main/notebooks/07_DeBERTa_FineTuning.ipynb)

This notebook fine-tunes **`microsoft/deberta-v3-base`** on the IMDb dataset.
DeBERTa (Decoding-enhanced BERT with disentangled attention) improves upon standard BERT and DistilBERT via:
1. **Disentangled Attention**: Words and relative positions are represented with distinct vectors.
2. **Enhanced Mask Decoder**: Incorporates absolute word positions right before prediction.
3. **Smart Head + Tail Truncation**: Preserves both the movie premise (first 128 tokens) and the reviewer's concluding verdict (last 382 tokens) instead of naive front-truncation.

> **Hardware Recommendation**: Run this on a GPU runtime (Google Colab T4/V100 or Kaggle GPU P100/T4). In Colab, go to `Runtime > Change runtime type > T4 GPU`.


In [ ]:
# 1. Hardware Verification
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: Running on CPU. Training will be very slow. Please enable a GPU runtime!")


In [ ]:
# 2. Dependency Installation (Required for DeBERTa-v3 & Hugging Face)
!pip install -q transformers datasets accelerate scikit-learn sentencepiece


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)


### 3. Automated Dataset Ingestion
This cell loads `train_reviews_clean.csv`. If running in Google Colab where files are not yet present, it automatically clones your GitHub repository (`--depth 1`) in 3 seconds to fetch the exact cleaned dataset used by your baseline and DistilBERT models.


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

csv_path = "train_reviews_clean.csv"

# Check if file exists locally, in parent dir, or clone from repo
if not os.path.exists(csv_path):
    if os.path.exists("../data/processed/train_reviews_clean.csv"):
        csv_path = "../data/processed/train_reviews_clean.csv"
    elif os.path.exists("IMDB-Sentiment-Research-Project/data/processed/train_reviews_clean.csv"):
        csv_path = "IMDB-Sentiment-Research-Project/data/processed/train_reviews_clean.csv"
    else:
        print("Fetching processed dataset from GitHub repository...")
        !git clone --depth 1 https://github.com/Ranjitcodes1/IMDB-Sentiment-Research-Project.git
        csv_path = "IMDB-Sentiment-Research-Project/data/processed/train_reviews_clean.csv"

print(f"Loading cleaned dataset from: {csv_path}")
df = pd.read_csv(csv_path)
df["clean_review"] = df["clean_review"].fillna("")

X = df["clean_review"].tolist()
y = df["label"].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Dataset successfully loaded!")
print(f"Training samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")


### 4. Smart Head + Tail Tokenizer (512 Max Length)
Standard truncation removes the end of reviews. Movie reviews typically reveal the decisive rating/verdict in the final sentence.
Here, we take the **first 128 tokens** and the **last 382 tokens** for any review exceeding 512 tokens.


In [ ]:
MODEL_CHECKPOINT = "microsoft/deberta-v3-base"
print(f"Loading tokenizer: {MODEL_CHECKPOINT}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def smart_head_tail_encode(texts, max_length=512, head_len=128):
    """Retains the premise (first 128 tokens) and conclusion (last 382 tokens)."""
    tail_len = max_length - head_len - 2 # reserve 2 tokens for [CLS] and [SEP]
    all_input_ids = []
    all_attention_masks = []
    
    for text in texts:
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(tokens) <= (max_length - 2):
            ids = [tokenizer.cls_token_id] + tokens + [tokenizer.sep_token_id]
        else:
            head = tokens[:head_len]
            tail = tokens[-tail_len:]
            ids = [tokenizer.cls_token_id] + head + tail + [tokenizer.sep_token_id]
            
        all_input_ids.append(ids)
        all_attention_masks.append([1] * len(ids))
        
    return {"input_ids": all_input_ids, "attention_mask": all_attention_masks}

print("Tokenizing training and validation sets with head+tail truncation...")
train_encodings = smart_head_tail_encode(X_train)
val_encodings = smart_head_tail_encode(X_val)
print("Tokenization completed successfully!")


In [ ]:
class IMDbTorchDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbTorchDataset(train_encodings, y_train)
val_dataset = IMDbTorchDataset(val_encodings, y_val)


### 5. Initializing DeBERTa-v3 Architecture


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2
)
print("DeBERTa-v3 architecture initialized!")


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }


### 6. Training Arguments & Optimization Recipe
* **Learning Rate**: `2e-5` with linear warmup and cosine decay.
* **Effective Batch Size**: 16 (`per_device_train_batch_size=8`, `gradient_accumulation_steps=2`).
* **FP16 Mixed Precision**: Accelerates GPU throughput.
* **Early Stopping**: Halts training if validation accuracy does not improve after 2 evaluations.


In [ ]:
# Setup Training Arguments
# Note: warmup_steps is used instead of warmup_ratio for cross-version compatibility
training_args = TrainingArguments(
    output_dir="./deberta_v3_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=200,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


### 7. Launching Fine-Tuning


In [ ]:
# Execute fine-tuning (takes ~20-25 minutes on free Google Colab T4 GPU)
trainer.train()


### 8. Final Evaluation & Classification Report


In [ ]:
eval_results = trainer.evaluate()
print("Final Evaluation Results:")
for k, v in eval_results.items():
    print(f"  {k}: {v}")

predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=["Negative", "Positive"], digits=4))


### 9. Exporting Model & Downloading Artifacts


In [ ]:
output_dir = "deberta_v3_imdb"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to '{output_dir}'.")

# Compress into a zip file for easy local download
!zip -r deberta_v3_imdb.zip deberta_v3_imdb/

# If running on Google Colab, trigger browser download:
try:
    from google.colab import files
    files.download("deberta_v3_imdb.zip")
    print("Browser download initiated!")
except ImportError:
    print("Download zip file manually from file explorer on the left.")
